In [1]:
from pathlib import Path
import csv
import glob
import random

import pandas as pd
import torch

In [2]:
GRAPH_PATH = "/scratch1/eibl/data/ukr_rus_twitter/graphs/retweet_graph_150files_minilm_hf03_political_labels.pt"
CSV_GLOB = "/project2/ll_774_951/uk_ru/twitter/data/*/*.csv"
SAMPLE_K = 30
SEED = 0
random.seed(SEED)

In [3]:
def load_interleaved_csv(filepath: str) -> pd.DataFrame:
    main_rows, sub_rows = [], []

    with open(filepath, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
            sub_header_raw = next(reader)
        except StopIteration:
            return pd.DataFrame()

    with open(filepath, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        next(reader)
        if sub_header_raw is not None:
            next(reader)

        pending_main = None
        for row in reader:
            if not row:
                continue
            if len(row) == 66:
                if pending_main is not None:
                    main_rows.append(pending_main)
                    sub_rows.append([""] * 11)
                pending_main = row
            elif len(row) == 11:
                if pending_main is not None:
                    main_rows.append(pending_main)
                    sub_rows.append(row)
                    pending_main = None

        if pending_main is not None:
            main_rows.append(pending_main)
            sub_rows.append([""] * 11)

    sub_cols = [
        "sub_extra",
        "state",
        "country",
        "rt_state",
        "rt_country",
        "qtd_state",
        "qtd_country",
        "norm_country",
        "norm_rt_country",
        "norm_qtd_country",
        "acc_age",
    ]
e
    df_main = pd.DataFrame(main_rows, columns=header)
    df_sub = pd.DataFrame(sub_rows, columns=sub_cols).drop(columns=["sub_extra"], errors="ignore")
    return pd.concat([df_main.reset_index(drop=True), df_sub.reset_index(drop=True)], axis=1)


def normalize_handle(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip().str.lower()
    return s.mask(s.isin(["", "nan", "none", "<na>"]))


In [4]:
# Load graph labels
raw = torch.load(GRAPH_PATH, map_location="cpu")
handles = raw["handles"]
y = raw["y"].cpu().numpy()
label_names = raw.get("label_names", ["left", "right"])

label_map = {}
for h, yi in zip(handles, y):
    if yi >= 0:
        label_map[h] = label_names[int(yi)]
    else:
        label_map[h] = "unlabeled"

print("graph nodes:", len(handles))
print("labeled nodes:", sum(yi >= 0 for yi in y))
print("label names:", label_names)

/home1/eibl/.conda/envs/prodigy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


graph nodes: 4428755
labeled nodes: 90430
label names: ['left', 'right']


In [ ]:
# Read first N raw files
N_FILES = 10
files = sorted(glob.glob(CSV_GLOB))[:N_FILES]
print("files:", len(files))
# print("first:", files[0])

dfs = []
for f in files:
    dfi = load_interleaved_csv(f)
    if not dfi.empty:
        dfi["source_file"] = Path(f).name
        dfs.append(dfi)

df = pd.concat(dfs, ignore_index=True)
df["screen_name_norm"] = normalize_handle(df["screen_name"])
df = df[df["screen_name_norm"].notna()].copy()

print("rows:", len(df))
print("unique accounts:", df["screen_name_norm"].nunique())


files: 20


In [ ]:
# One row per account, with graph label attached
acct = (
    df.sort_values("screen_name_norm")
      .drop_duplicates("screen_name_norm")
      .copy()
)

acct["graph_label"] = acct["screen_name_norm"].map(label_map).fillna("not_in_graph")

cols = [
    "screen_name_norm",
    "graph_label",
    "userid",
    "description",
    "source_file",
]


In [ ]:
sample_df = acct.groupby('graph_label').apply(lambda x: x[cols].sample(3)).reset_index(drop=True)
sample_df

In [ ]:
# df[df. sample_df.screen_name_norm]
# df[df.screen_name.eq(]
# df[df.screen_name_norm.isin(sample_df.screen_name_norm)]
# sample_df.screen_name_norm
# sample_df.description
for c in df[df.screen_name_norm.eq('aveclezlepen')].columns:
    # print(c)
    print(df[df.screen_name_norm.eq('aveclezlepen')][c])
    print()
    print()
# sample_df